In [1]:
import pandas as pd
import re

# --- CONFIGURATION ---
fichier_entree = "textes_bruts_tickets.csv"
fichier_sortie = "echantillon_propre_a_annoter.csv"

# --- RÈGLES DE NETTOYAGE (Ton code) ---
def est_une_impurete(texte):
    texte = str(texte).strip().upper()
    
    if len(texte) < 3:
        return True
        
    if re.match(r'^\d+([.,]\d+)?$', texte):
        return True
        
    if re.search(r'\d{2}/\d{2}/\d{2,4}', texte) or re.search(r'\d{2}:\d{2}(:\d{2})?', texte):
        return True
        
    # Liste des mots à bannir !
    mots_bannis = [
        "TOTAL", "TVA", "CARTE BANCAIRE", "CB", "MERCI", "VISITE", 
        "DUPLICATA", "SIRET", "CAISSE", "RENDU", "MONNAIE", "EUR", "TTC", "HT"
    ]
    if any(mot in texte for mot in mots_bannis):
        return True
        
    return False

# --- EXÉCUTION ---
print(f"Ouverture du gros fichier : {fichier_entree}...")
try:
    df = pd.read_csv(fichier_entree)
except FileNotFoundError:
    print(f"Erreur : Le fichier {fichier_entree} n'existe pas.")
    exit()

taille_initiale = len(df)

# 1. On applique TON nettoyage pour enlever les mots-clés et les dates
df_propre = df[~df['texte_brut'].apply(est_une_impurete)]

# 2. On enlève aussi ce qui a été détecté comme prix par EasyOCR
if 'est_un_prix' in df_propre.columns:
    df_propre = df_propre[df_propre['est_un_prix'] == False]

taille_nettoyee = len(df_propre)

# 3. L'échantillonnage pour te faire gagner du temps (300 lignes max)
if taille_nettoyee > 300:
    df_final = df_propre.sample(n=300, random_state=42)
else:
    df_final = df_propre

# On sauvegarde le résultat
df_final.to_csv(fichier_sortie, index=False, encoding='utf-8')

print("-" * 30)
print(f"Nettoyage et réduction terminés ! 🧹")
print(f"Lignes au départ : {taille_initiale}")
print(f"Lignes après nettoyage (vrais textes) : {taille_nettoyee}")
print(f"Lignes gardées pour l'annotation : {len(df_final)}")
print(f"Le fichier de travail est prêt : {fichier_sortie}")

Ouverture du gros fichier : textes_bruts_tickets.csv...
------------------------------
Nettoyage et réduction terminés ! 🧹
Lignes au départ : 19298
Lignes après nettoyage (vrais textes) : 10851
Lignes gardées pour l'annotation : 300
Le fichier de travail est prêt : echantillon_propre_a_annoter.csv
